In [3]:
# prompt: Quisiera importar la libreria optuna

!pip install optuna
!pip install -U kaleido
!pip install optuna-dashboard
#!pip install kaleido


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 6.5 MB/s eta 0:00:00


In [1]:
####IMPORTANTE CARGAR UTILS.PY
from google.colab import files
uploaded = files.upload()

Saving utils.py to utils.py


In [4]:
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Definir la ruta base para tu Google Drive
BASE_DIR_DRIVE = '/content/drive/MyDrive'
PATH_TO_TRAIN = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/input/petfinder-adoption-prediction/train/train.csv")
# Artefactos a subir a optuna
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_temp_artifacts")
# Artefactos que optuna gestiona
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_artifacts")

In [15]:
# study_lgb = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name="04 - LGB Multiclass CV",
#                             load_if_exists = True)

# study_lgb = optuna.create_study(
#     direction='maximize',
#     storage="sqlite:///work/db.sqlite3",
#     study_name="04 - LGB Multiclass CV",
#     load_if_exists=True
# )

ruta_carpeta_work = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'

# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db = os.path.join(ruta_carpeta_work, 'db.sqlite3')

# Ahora utiliza esta ruta en la configuración de Optuna
storage = f"sqlite:///{ruta_completa_db}"

study_lgb = optuna.create_study(
    direction='maximize',
    storage=storage,
    study_name="04 - LGB Multiclass CV",
    load_if_exists=True
)



lgb_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

[I 2025-04-26 23:54:57,120] Using an existing study with name '04 - LGB Multiclass CV' instead of creating a new one.


In [16]:
lgb_dataset

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed,pred
14696,1,Dione & Elora,1,307,307,2,1,0,0,2,...,2,0,41327,61b07b54adb97d4b5f3c2dec06a9943b,0,Dione and Elora are puppies of Rambo. Both are...,8f20e24ef,9.0,4,"[0.08548294021191288, 0.8233842391290679, 1.96..."
14823,1,Har-nee,24,103,307,2,1,2,4,2,...,1,0,41330,9cb2e5a10e24e0b09942013b8434c81f,0,We found Har-nee with a swollen and almost sev...,2d72ef0c4,2.0,4,"[0.11135507955265758, 0.7921538780973132, 1.03..."
2838,1,The Gorgeous 5 Beauties,2,307,0,2,2,7,0,2,...,5,0,41326,5c398b2e18b16f0db83c53e682eada42,0,Theses 5 very adorably cute white female puppi...,44cd12263,5.0,4,"[0.05843595583061092, 0.5410861706358654, 1.69..."
1848,2,Mochi,1,265,0,1,2,0,0,1,...,1,0,41401,6905e4fbe5658eef5f560b814898a5ee,2,Hello! My name is Mochi. I was rescued from a ...,210c4a637,6.0,2,"[0.16397517379338017, 1.1821686606966122, 2.03..."
669,2,Nala & Peach,9,266,266,2,2,4,6,2,...,2,0,41326,803457cd3660dda694086b51a11a5a39,0,Nala is a cat that's been born with 7 fingers ...,21493e6ea,8.0,4,"[0.1023048583929651, 0.6655116636835393, 1.457..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,2,Anak Nanya,8,266,0,3,1,2,0,2,...,3,50,41326,f14c2cfebbbafbc9ed1f500d082f3ec3,0,they r ol so cute :) it juz a matter me dun hv...,35f9818a7,14.0,4,"[0.0579129811364184, 0.6312486278355145, 0.979..."
12222,1,Poor Baby,3,307,0,1,5,0,0,2,...,1,0,41401,500c48db7b281eabec3c293160f4a71c,0,On behalf of Exotica Pets Healthy puppy availa...,46e25aa2b,2.0,1,"[0.060087608237161444, 1.4013187290352689, 1.5..."
10538,2,No Name,1,265,0,2,1,6,0,1,...,1,0,41401,ac9a633cf51a70f4a9842e6e1ba91fc9,0,sy jumpa kitten ni mengiau2 kat playground. ra...,d3692d2b2,2.0,1,"[0.1377683026157045, 1.8928723191890653, 1.494..."
11062,1,Pipi,1,307,0,2,1,5,7,2,...,6,0,41326,3ef66c1034bb6dc31314845457079483,0,"Health, cute and active puppies.",3c43b7541,1.0,4,"[0.07516875569269361, 0.6859579709863249, 1.77..."


In [17]:
MODEL_NAME = '06 Bert'
MODEL_VERSION = '1.0'

# study_bert = optuna.create_study(direction='maximize',
#                             storage="sqlite:///work/db_bert.sqlite3",  # Specify the storage URL here.
#                             study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
#                             load_if_exists = True)

# Define la ruta a la carpeta 'work' en tu Google Drive
ruta_carpeta_work_bert = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'

# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db_bert = os.path.join(ruta_carpeta_work_bert, 'db_bert.sqlite3')

# Ahora utiliza esta ruta en la configuración de Optuna
storage_bert = f"sqlite:///{ruta_completa_db_bert}"



study_bert = optuna.create_study(direction='maximize',
                                 storage=storage_bert,
                                 study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                                 load_if_exists = True)

bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

[I 2025-04-26 23:56:57,489] Using an existing study with name '06 Bert_1.0' instead of creating a new one.


In [18]:
bert_dataset

,PetID,pred,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PhotoAmt,AdoptionSpeed,labels
0,8e76c8e39,"[2.4382503e-05, 0.00010504124, 0.0008790736, 0...",2,Kali,3,264,0,2,1,2,...,1,1,50,41326,a9caef3f98e67bfac9093cca79e20b93,0,Kali is a super playful kitten who is on the g...,2.0,1,1
1,6436c1a59,"[2.4075793e-05, 0.0001411934, 0.058392707, 0.9...",1,Godiva,12,307,0,2,2,7,...,1,1,0,41326,a042471e0f43f2cf707104a1a138a7df,0,Godiva was rescued in Serdang residential area...,7.0,2,2
2,988988d5b,"[4.9734066e-05, 0.00092424767, 0.008126355, 0....",2,Cikenet,3,266,0,1,2,7,...,1,1,0,41401,b8853c71b981104f1ef126e51387b616,0,"hello cikenets fans, i just wanna inform that ...",19.0,1,1
3,efbf1703a,"[0.0014253455, 0.02513897, 0.79662913, 0.17567...",2,No Name,1,266,0,2,1,0,...,1,1,0,41326,2f846fb8f87a25678374e193559d83c9,0,"Just saved this kitten from the street, but i ...",2.0,2,2
4,543130f60,"[1.1735334e-05, 0.00089636375, 0.9900468, 0.00...",1,BoiBoi,24,307,0,1,5,7,...,1,5,0,41326,2147467fcd35e7a3bc23b9edcffc5702,0,Boiboi is rescued by my daughter 2 years ago f...,1.0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2991,23874f644,"[0.00028118477, 0.0018251932, 0.9935302, 0.000...",1,Patch,8,307,0,2,2,7,...,1,1,0,41326,001e42763024f9d4abe31e79472b1827,0,Patch is for free adoption. If you want to ado...,2.0,3,3
2992,e7f7066b6,"[0.0038137618, 0.18633553, 0.02154754, 0.62215...",1,Terry,24,179,307,1,2,3,...,1,1,0,41326,719987dce7aeb027fdfa91b480800199,0,been at my place for a while..am hoping to fin...,0.0,4,4
2993,36e7f8d83,"[8.308675e-06, 0.00030206467, 0.003118829, 0.9...",2,Pets + Strays : BlueEyed BlackWhite,1,266,0,2,5,6,...,1,1,0,41401,90569c3f7cb0af35cba5dac82c0ac9d7,0,1 month old white + grey kitten for adoption n...,1.0,3,3
2994,4d163b731,"[0.00109846, 0.22724706, 0.5555739, 0.19682041...",1,Snowy,6,195,0,2,1,7,...,1,1,0,41401,79309f4027f2fedb4349a298c69fe56f,0,ooooo,1.0,0,0


In [19]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')



merged_datasets['bert_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['bert_pred_score'] ]

In [20]:
merged_datasets

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score
0,002230dea,"[0.1745226372347884, 1.4702635764749898, 1.941...",1,"[1.7439721e-05, 0.009094744, 0.9885337, 0.0023..."
1,0063f83c9,"[0.29659562842443415, 1.1094881868202284, 1.23...",1,"[0.005029747, 0.15550436, 0.51722467, 0.318468..."
2,0073c33d0,"[0.06463103526891116, 0.9812651347903425, 1.67...",3,"[9.053005e-05, 0.005007521, 0.012728454, 0.981..."
3,00bfa5da9,"[0.0669623048794452, 0.5757829703218922, 1.477...",4,"[1.4863421e-05, 5.312055e-05, 0.000103065795, ..."
4,00c19f4fa,"[0.06198863768641051, 0.7241736821971771, 1.53...",2,"[5.828552e-05, 0.00072690164, 0.99359137, 0.00..."
...,...,...,...,...
2994,ffa5c6c35,"[0.215835409551889, 0.9135602252764407, 1.0222...",4,"[0.0037388061, 0.029114172, 0.109155364, 0.148..."
2995,ffd697903,"[0.21320604627408152, 1.3282664936333566, 1.71...",3,"[5.006563e-05, 0.08872734, 0.8981254, 0.010486..."
2996,ffe0f06ab,"[0.12309816370957066, 1.5266719005548763, 1.62...",2,"[0.00040289157, 0.08617168, 0.06141095, 0.8511..."
2997,ffe5a0271,"[0.08496250693791985, 1.6652229164206878, 1.56...",3,"[2.0663645e-05, 0.08348679, 0.12993942, 0.7859..."


In [21]:
merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score'] for i,r in merged_datasets.iterrows()]

In [22]:
merged_datasets['lgb_pred_score']

,lgb_pred_score
0,"[0.1745226372347884, 1.4702635764749898, 1.941..."
1,"[0.29659562842443415, 1.1094881868202284, 1.23..."
2,"[0.06463103526891116, 0.9812651347903425, 1.67..."
3,"[0.0669623048794452, 0.5757829703218922, 1.477..."
4,"[0.06198863768641051, 0.7241736821971771, 1.53..."
...,...
2994,"[0.215835409551889, 0.9135602252764407, 1.0222..."
2995,"[0.21320604627408152, 1.3282664936333566, 1.71..."
2996,"[0.12309816370957066, 1.5266719005548763, 1.62..."
2997,"[0.08496250693791985, 1.6652229164206878, 1.56..."


In [23]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [24]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [25]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred'],
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred'],
                                                                    weights='quadratic')))

In [26]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred'],
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['bert_pred'],
                                                                    weights='quadratic')))



In [27]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blended_pred'],
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['blended_pred'],
                                                                    weights='quadratic')))
